# Job Salary Estimator — DAY 5: Fine-tuning a Frontier Model

Fine-tune GPT-4.1-nano (or similar) on job→salary examples via OpenAI API.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from job_salary.items import Job
from job_salary.evaluator import evaluate

load_dotenv(override=True)
openai_client = OpenAI()

In [ ]:
# Load data
try:
    train, val, test = Job.from_hub('ed-donner/job_salaries_processed')
except Exception:
    from datasets import load_dataset
    from job_salary.parser import parse
    from tqdm.notebook import tqdm
    ds = load_dataset('lukebarousse/data_jobs', split='train', trust_remote_code=True)
    items = [parse(dp) for dp in tqdm(ds.select(range(10000)))]
    items = [j for j in items if j is not None]
    n = len(items)
    train, val, test = items[:int(0.8*n)], items[int(0.8*n):int(0.9*n)], items[int(0.9*n):]

def get_text(job):
    return job.summary if job.summary else job.full or ''

ft_train = train[:100]   # OpenAI recommends 50-100 to start
ft_val = val[:50]
print(f'Fine-tune train: {len(ft_train)}, val: {len(ft_val)}')

In [ ]:
def messages_for(job):
    text = get_text(job)
    return [
        {"role": "user", "content": f"Estimate the yearly salary in USD. Respond with only the number.\n\n{text}"},
        {"role": "assistant", "content": f"${job.salary:.0f}"}
    ]

def make_jsonl(jobs, path):
    os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
    with open(path, 'w') as f:
        for j in jobs:
            line = json.dumps({"messages": messages_for(j)})
            f.write(line + '\n')

make_jsonl(ft_train, 'jsonl/fine_tune_train.jsonl')
make_jsonl(ft_val, 'jsonl/fine_tune_validation.jsonl')

In [ ]:
# Upload files to OpenAI
with open('jsonl/fine_tune_train.jsonl', 'rb') as f:
    train_file = openai_client.files.create(file=f, purpose='fine-tune')

with open('jsonl/fine_tune_validation.jsonl', 'rb') as f:
    val_file = openai_client.files.create(file=f, purpose='fine-tune')

print(f'Train file: {train_file.id}, Val file: {val_file.id}')

In [ ]:
# Create fine-tuning job
job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model='gpt-4.1-nano-2025-04-14',
    seed=42,
    hyperparameters={'n_epochs': 1, 'batch_size': 1},
    suffix='salary'
)
print(f'Job ID: {job.id}')
print('Monitor at: https://platform.openai.com/finetune')

In [ ]:
# After job completes, retrieve model name
status = openai_client.fine_tuning.jobs.retrieve(job.id)
ft_model = status.fine_tuned_model if status.fine_tuned_model else None
print(f'Status: {status.status}, Model: {ft_model}')

In [ ]:
# Inference with fine-tuned model
def ft_pricer(job):
    if not ft_model:
        return 0
    r = openai_client.chat.completions.create(
        model=ft_model,
        messages=[{"role": "user", "content": f"Estimate yearly salary in USD. Respond with only the number.\n\n{get_text(job)}"}],
        max_tokens=10
    )
    return r.choices[0].message.content

evaluate(ft_pricer, test, size=min(100, len(test)))